<a href="https://colab.research.google.com/github/Satyadeep-Dey/genai-lab/blob/main/14_1_Use_case_Resume_RAG_using_semantic_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resume RAG using semantic search :

**Data source :** resumes from Hugging Face - opensporks/resumes

**Improvement from the previous RAG experiments :**

Creating chunks based on sections instead of dividing the document into chunks of equal size .

As a result each resume will be broken into multiple chunks with each section (e.g., Skills, Certifications etc.) being represented by 1 or more chunks.
So a single resume produces multiple vectors which are stored in the same vector database, along with metadata such as candidate_id,
section_type etc.

With section-level embeddings:
*  a query about skills matches skill vectors directly
*  a query about experience matches experience vectors directly

retrieval becomes more precise and less noisy.

**Note:**
For section-based chunking, overlap is generally not needed as each chunk is already a semantically complete unit. Exception : very long sections that are split across multiple chunks e.g., experience . We do overlap for those only.


In [ ]:
!pip install -q openai datasets huggingface_hub pdfplumber faiss-cpu

In [ ]:
#Imports
from openai import OpenAI
from google.colab import userdata
from datasets import load_dataset
import pdfplumber
import re
import torch
import sys
from google.colab import drive
from datasets import load_from_disk
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

# Execute below cell only once ..
to load the dataset from Hugging Face
and save it in Hugging Face native format

**save_to_disk is better than pickle because:**

*   Designed specifically for HuggingFace Dataset objects — preserves
schema, column types, and metadata exactly
*   Uses Arrow format internally, so loading is very fast (memory-mapped, no full deserialization)
*   Handles large datasets without loading everything into RAM at once
*   Works correctly with DatasetDict (train/test/validation splits) out of the box

**Once saved to Google Drive, we can simply upload**

In [ ]:
# import sys
# from google.colab import drive

# dataset = load_dataset("opensporks/resumes")

# # Mount Google Drive
# drive.mount('/content/drive')
# dataset.save_to_disk("/content/drive/MyDrive/Files/Knowledge-Base/resume_rag/resumes_100")


In [ ]:
# Mount Google Drive
drive.mount('/content/drive')


**You can jump to the cell where vector embedding , chunks and meta-data are loaded from Google Drive if those tasks were already done in a previous execution**

In [ ]:
#Load dataset from Google Drive
dataset = load_from_disk(
    "/content/drive/MyDrive/Files/Knowledge-Base/resume_rag/resumes_100"
)

In [ ]:
# Number of resumes
num_resumes = len(dataset['train'])
print("Number of resumes:", num_resumes)

In [ ]:
# get the total dataset size on disk
!du -sh /content/drive/MyDrive/Files/Knowledge-Base/resume_rag/resumes_100

In [ ]:
sample = dataset['train'][0]

pdf = sample['pdf']

full_text = ""

for page in pdf.pages:
    page_text = page.extract_text()

    if page_text:
        full_text += page_text + "\n"

print(full_text)


**Prepare this for RAG**

In [ ]:
#1. Extract text from PDFs

def extract_text(example):
    pdf = example["pdf"]

    full_text = ""

    for page in pdf.pages:
        text = page.extract_text()

        if text:
            full_text += text + "\n"

    return {
        "text": full_text,
        "label": example["label"]
    }


In [ ]:
text_dataset = dataset['train'].map(
      extract_text,
      remove_columns=["pdf"]
      ) #Load full datset = 2484 resumes

# small_dataset = dataset['train'].select(range(500)) # get first 500 resumes
# text_dataset = small_dataset.map(
#     extract_text,
#     remove_columns=["pdf"]   # IMPORTANT
# )

#pdfplumber.PDF objects are not Arrow-serializable
#remove_columns=["pdf"] drops them before Hugging Face writes the new dataset
#resulting dataset contains only text and label
# which are clean serializable fields suitable for RAG pipelines.

In [ ]:
print(f"Number of resumes in dataset is : {len(text_dataset)}")
print("----------------------------")
print(f"Here is a sample resume : {text_dataset[0]["text"]}")

In [ ]:
# Step 2 : Clean the text
def clean_text(text):
    text = text.replace("ï¼​", "").replace("Â", "").replace("â€‹", "")
    text = re.sub(r'[^\S\n]+', ' ', text)   # collapse spaces/tabs but keep newlines
    text = re.sub(r'\n{3,}', '\n\n', text)  # max 2 consecutive newlines
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return text.strip()

In [ ]:
sample = clean_text(text_dataset[0]["text"])
print(f"Here is first 1000 characters of a sample resume : {sample[:1000]}")
#print(sample[:1000])

In [ ]:
# Step 3 : Chunking

SECTION_HEADERS = [
    "Summary", "Objective", "Experience", "Education",
    "Skills", "Certifications", "Highlights", "Accomplishments",
    "Additional Information", "Interests"
]

def chunk_resume_by_section(text):
    pattern = r'(?=^(' + '|'.join(SECTION_HEADERS) + r')\s*$)'
    parts = re.split(pattern, text, flags=re.MULTILINE)

    chunks = []
    current_header = "Header"
    current_body = []

    for part in parts:
        part = part.strip()
        if not part:
            continue
        if part in SECTION_HEADERS:
            if current_body:
                chunks.append({
                    "section": current_header,
                    "text": " ".join(current_body)
                })
            current_header = part
            current_body = []
        else:
            current_body.append(part)

    if current_body:
        chunks.append({
            "section": current_header,
            "text": " ".join(current_body)
        })

    return chunks


In [ ]:
# 1.Ensure that chunks are not too big or too small

# 1.1. all-MiniLM-L6-v2 has a 256 token limit (~200 words).
# Anything beyond that gets silently truncated during embedding,
# so most of that resume's experience would never be retrievable.

# 1.2. Header chunks (1-2 words) are noise
#They just contain the job category label and add nothing useful to retrieval.
# Here's the fix — add sub-chunking for long sections and filter out short ones

MAX_WORDS = 200  # safe limit for all-MiniLM-L6-v2
MIN_WORDS = 10   # filter out noise chunks
OVERLAP = 30

def sub_chunk(text, max_words=MAX_WORDS, overlap=OVERLAP):
    words = text.split()

    # No sub-chunking needed, return as-is
    if len(words) <= max_words:
        return [text]

    # Apply overlap only when splitting is necessary
    return [
        " ".join(words[i:i + max_words])
        for i in range(0, len(words), max_words - overlap)
    ]



In [ ]:
#In the Opensporks dataset, labels are numeric category IDs.
# something like:accountant, HR, engineer, healthcare
# this is very useful info for RAG search

labels = dataset["train"].features["label"].names
for label in labels:
  print(label)

range_start = 100
number_of_resumes = 15

for i, row in enumerate(text_dataset.select(range(range_start,range_start + number_of_resumes))):
    label_text = labels[row["label"]]
    print(f"Resume #{range_start+ i} | {label_text}")

In [ ]:
# Build flat lists ready for embedding
all_chunks = []
all_metadata = []

for i, row in enumerate(text_dataset):
    label_text = labels[row["label"]] # get text category not number # row["label"]
    cleaned = clean_text(row["text"])
    sections = chunk_resume_by_section(cleaned)

    for section in sections:
        text = section["text"]

        # Skip noise chunks
        if len(text.split()) < MIN_WORDS:
            continue

        # Sub-chunk long sections
        sub_chunks = sub_chunk(text)
        for j, sub in enumerate(sub_chunks):
            all_chunks.append(f"{label_text}\n{sub}")  # label prepended here # all_chunks.append(sub)
            all_metadata.append({
                "resume_id": i,
                "identifier": f"{label_text} | Resume #{i}",
                # "category": label_text, # duplicate of identifier
                "section": section["section"],
                "sub_chunk": j
            })

In [ ]:
print(f"Total chunks: {len(all_chunks)}")
print(f"Avg chunks per resume: {len(all_chunks) / len(text_dataset):.1f}")

In [ ]:
# Let's check some of these chunks

for i, chunk in enumerate(all_chunks[:25]): # first 25 chunks
    resume_id = all_metadata[i]['resume_id']
    print(f"Resume #{resume_id} | Chunk {i} | {len(chunk.split())} words | {chunk[:200]}")


In [ ]:
# Let's check some of these metadata
# for i, metadata in enumerate(all_metadata[:25]):
#     print(f"Index: {i} | resume_id: {metadata['resume_id']} | section: {metadata['section']} | sub_chunk: {metadata['sub_chunk']}")

for i, meta in enumerate(all_metadata[:25]):
    print(f"Resume #{meta['resume_id']} | Chunk {i} | Identifier : {meta['identifier']} | Section : {meta['section']} | Sub-chunk: {meta['sub_chunk']} | {len(all_chunks[i].split())} words" )
    #print(meta)


In [ ]:
# Step 4 : Embedding

# Note : Added a couple of parameters to model.encode() even though they are
# default values , just to be on the safe side

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True # default in SentenceTransformer
).astype("float32") #model.encode() returns float32 by default

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

In [ ]:
print(f"Embeddings shape: {embeddings.shape}")
print(f"Vectors in index: {index.ntotal}")

In [ ]:
# Save everything to Google Drive so you don't have to rebuild it each session:

# Save FAISS index
faiss.write_index(index,"/content/drive/MyDrive/Files/Knowledge-Base/resume_rag/index.faiss")

# Save chunks and metadata
with open("/content/drive/MyDrive/Files/Knowledge-Base/resume_rag/chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

with open("/content/drive/MyDrive/Files/Knowledge-Base/resume_rag/metadata.pkl", "wb") as f:
    pickle.dump(all_metadata, f)

print("Saved successfully")


In [ ]:
#Reload in a future session without rebuilding:

from sentence_transformers import SentenceTransformer

index = faiss.read_index("/content/drive/MyDrive/Files/Knowledge-Base/resume_rag/index.faiss")

with open("/content/drive/MyDrive/Files/Knowledge-Base/resume_rag/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

with open("/content/drive/MyDrive/Files/Knowledge-Base/resume_rag/metadata.pkl", "rb") as f:
    all_metadata = pickle.load(f)

model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Index loaded: {index.ntotal} vectors")
print(f"Chunks loaded: {len(all_chunks)}")


In [ ]:
#Step 5: Setup an LLM that can answer questions based on Context

#define constants
MODEL = "gpt-4.1-mini"
#MODEL = "gpt-5.2"

# Sign in to OpenAI using Secrets in Colab
openai_api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=openai_api_key)

# Check if Open AI key has been set
if openai_api_key:
    print("OpenAI API Key exists")
else:
    print("OpenAI API Key not set")

In [ ]:
# Since you're using IndexFlatIP with normalized vectors,
# the scores are cosine similarity values between -1 and 1:
# Score range	Meaning
# 0.7 — 1.0	Very strong match
# 0.5 — 0.7	Good match
# 0.3 — 0.5	Weak but related
# Below 0.3	Likely irrelevant
# Start with threshold=0.3 and tune up or down based on the quality of results you get.
# You can also do a quick test to see what scores your queries are actually returning

# from sentence_transformers import CrossEncoder
# reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve(query, k=20, threshold=0.3):
    q_emb = model.encode([query])
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if score >= threshold:
            results.append({
                "chunk": all_chunks[idx],
                "metadata": all_metadata[idx],
                "score": float(score)
            })

    # Deduplicate — keep only the highest scoring chunk per resume
    seen = set()
    deduped = []
    for r in results:
        rid = r['metadata']['resume_id']
        if rid not in seen:
            seen.add(rid)
            deduped.append(r)

    return deduped[:5]  # return top 5


In [ ]:
def rag_query(question,k=20, threshold=0.3):

    results = retrieve(question,k,threshold)

    if not results:
        return {"answer": "No relevant results found.", "sources": []}

    context_parts = [
        f"[Source: {r['metadata']['identifier']} | {r['metadata']['section']}]\n{r['chunk']}"
        for r in results
    ]

    context = "\n\n---\n\n".join(context_parts)

    message = [
            {
                "role": "system",
                "content": (
                    "You answer questions about resumes using only the provided context. "
                    "Cite the source for each piece of information you use. "
                    "If the question cannot be answered from the context, reply with exactly: DO-NOT-KNOW"
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ]

    response = client.chat.completions.create(
        model= MODEL,
        max_tokens=250, #controls API costs
        messages=message
    )

    answer = response.choices[0].message.content

    if answer.strip() == "DO-NOT-KNOW":
        return {"answer": "I could not find relevant information in the resume database.", "sources": []}

    sources = [
        {
            "identifier": r['metadata']['identifier'],
            "section": r['metadata']['section'],
            "text": r['chunk'],
            "score":  r['score']
        }
        for r in results
    ]

    return {"answer": answer, "sources": sources}

In [ ]:
# Function to get answer along with source
import re
from IPython.display import display, Markdown

def ask(question,k=20, threshold=0.3):
    result = rag_query(question,k,threshold)
    md = f"## Answer\n{result['answer']}\n\n"
    if result["sources"]:
        md += "## Sources\n"
        for s in result["sources"]:
          md += f"\n**Category : {s['identifier']} | Section : {s['section']} | Score: {s['score']:.3f}**\n\n**Text:** {s['text'][:400]}\n"
          # \n is a newline — moves to the next line
          # ** is Markdown bold syntax — everything between two ** pairs renders as bold text

    display(Markdown(md))

In [ ]:
ask("Who has experience with accounts payable?")#Specific Skill

In [ ]:
ask("Who has experience with ERP systems?")#Specific Skill

In [ ]:
ask("Who has experience with managed enterprise systems ?")#different name that could mean ERP !

Below question illusrates an important point . The LLM may suggest a candidate with lower score .

This happens because FAISS score measures how similar the chunk text is to your query.

The LLM does something different — it reasons across all retrieved chunks and evaluates which candidate satisfies ALL requirements.When GPT reads the full context it notices:

*  #1586 — strong leadership, but no ERP mentioned
*  #1661 — strong leadership, but no ERP mentioned
*  #1582 — good leadership AND specific ERP systems listed (Oracle, PeopleSoft, SAP)

So GPT correctly picks #1582 as the best fit for the combined requirement, even though its FAISS score is lower.

This is exactly the two-layer value of RAG:

*  FAISS — fast broad retrieval, finds relevant candidates
*  LLM — slow deep reasoning, picks the best one

The higher FAISS score just means "most similar text to your query" — not "best answer to your question".




In [ ]:
ask("best candidate for a senior finance role requiring leadership and ERP")

In [ ]:
ask("Who has payroll processing experience?")#Specific Skill

In [ ]:
ask("Who has experience with government accounting?")#Specific Skill

In [ ]:
ask("Who has both management and accounting experience?")#Multi-hop

In [ ]:
ask("Who has supervised large teams and managed budgets?")#Multi-hop

In [ ]:
ask("Who has experience with Python or machine learning?")#Multi-hop

In [ ]:
ask("Who has a PhD?")#Negative Test

In [ ]:
q = """
Which candidate is a Summary Graduating Ph.D. candidate with a
research focus on developing large-scale computational models using statistics?
"""
ask(q) # if question becomes more specific then it finds the candidate

In [ ]:
ask("Who is the most experienced candidate?") #Ambiguous - does not give correct answer

In [ ]:
ask("Who would be best for a senior finance role?")#Ambiguous

In [ ]:
ask("What certifications do candidates have?")#Section Specific

In [ ]:
ask("What are the educational qualifications of the candidates?")#Section Specific

In [ ]:
ask("what is the capital of India ?") # question not in knowledge base and hence no answer

In [ ]:
ask("show me Resume #100 ?")
# will not find an answer
# Because FAISS does semantic similarity search — it compares the meaning of our
# query against the meaning of chunk text.
# "Resume #100" has no semantic meaning that matches anything in the chunks.

In [ ]:
ask("How many resumes of agriculture ?")
# cannot tell how many but lists 5 since we're returing top 5

# Find resumes based on job descriptions

We're using SentenceTransformer for embedding and it silently truncates as it has a 256 token limit.

Hence we'll first summarize the job description and then seach resumes .

In [ ]:
def extract_requirements(job_description):
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=250,
        messages=[{
            "role": "user",
            "content": (
                f"Extract the skills and requirements from this job description to fit into maximum 250 tokens."
                f"Answer as a short comma-separated list. No explanation, just the list.\n\n{job_description}"
            )
        }]
    )
    return response.choices[0].message.content

In [ ]:
def ask_with_jd(job_description,k=20, threshold=0.3):
    requirements = extract_requirements(job_description)
    print(f"Extracted requirements: {requirements}\n")
    question = f"Who has experience with {requirements}?"
    return ask(question,k,threshold)


In [ ]:
JD = """
We are looking for a Software Engineer with experience in Python, Java, SQL, REST APIs, cloud technologies, and web application development.

Responsibilities:
- Build scalable backend systems
- Work with databases and APIs
- Collaborate with cross-functional teams
- Participate in Agile development

Required Skills:
Python, Java, SQL, AWS, Git, Linux, REST APIs, JavaScript

Preferred:
Docker, Kubernetes, React
"""
requirements = extract_requirements(JD)
number_of_words = len(requirements.split())
print(f"Number of words : {number_of_words}")
print("----------------------------------")
print(requirements)


In [ ]:
# useful for debugging and finertuning threshold
results = retrieve(requirements, k=20, threshold=0.0)
for r in results:
    print(f"Score: {r['score']:.3f} | {r['metadata']['identifier']} | {r['metadata']['section']}")

In [ ]:
ask_with_jd(requirements)

In [ ]:
JD = """
Seeking a Data Analyst proficient in SQL, Excel, Power BI, Tableau, and Python.

Responsibilities:
- Build dashboards
- Analyze business metrics
- Create reports for stakeholders
- Perform data cleaning and visualization

Skills:
SQL, Excel, Tableau, Power BI, Python, Statistics
"""
requirements = extract_requirements(JD)
ask_with_jd(requirements)

In [ ]:
JD = """
We are hiring an HR Recruiter with experience in talent acquisition, onboarding, employee engagement, and payroll coordination.

Requirements:
- Experience with recruitment lifecycle
- Interview scheduling
- HR operations
- Employee relations

Skills:
Recruitment, HRMS, onboarding, payroll, communication
"""
requirements = extract_requirements(JD)
ask_with_jd(requirements)

In [ ]:
JD = """
Looking for a Business Development Manager to drive client acquisition and revenue growth.

Responsibilities:
- Lead generation
- Client relationship management
- Sales strategy
- Market research

Skills:
CRM, sales pipeline, negotiation, business strategy
"""

requirements = extract_requirements(JD)
ask_with_jd(requirements)

In [ ]:
JD = """
Seeking an Accountant with experience in financial reporting, taxation, accounts payable/receivable, and auditing.

Skills:
Tally, GST, SAP, QuickBooks, Excel, financial analysis
"""

requirements = extract_requirements(JD)
ask_with_jd(requirements)

In [ ]:
JD = """
Looking for a Mechanical Engineer with CAD design and manufacturing experience.

Skills:
AutoCAD, SolidWorks, production planning, quality assurance
"""

requirements = extract_requirements(JD)
ask_with_jd(requirements)

In [ ]:
JD = """
Seeking a Mathematics Teacher with classroom management and curriculum planning experience.

Skills:
Lesson planning, classroom teaching, assessment, communication
"""

requirements = extract_requirements(JD)
ask_with_jd(requirements)